# PathSnap: GPS Map-Matching & Route Reconstruction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gagan-1718/PathSnap-GPS-Map-Matching-Route-Reconstruction/blob/feat/pathsnap-pipeline/PathSnap_Colab.ipynb)

This notebook runs the project end to end, using the code in this repository:

1. Setup and data
2. Exploratory data analysis
3. Candidate generation and features
4. Training 5 models (GroupKFold CV by trip)
5. Student-test evaluation
6. Bayesian hyperparameter optimisation
7. HMM + Viterbi route reconstruction
8. Folium maps
9. API pipeline call

All numbers shown are produced by the cells below. Nothing is hardcoded.

## 1. Setup

In [ ]:
!git clone -q -b feat/pathsnap-pipeline https://github.com/Gagan-1718/PathSnap-GPS-Map-Matching-Route-Reconstruction pathsnap
%cd pathsnap
!pip install -q optuna folium

### Dataset
The raw data is not on GitHub (about 90 MB). Upload the provided **`Student folder.zip`**. It contains `train/` (80 trips) and `student_test/` (10 trips).

In [ ]:
import os, shutil, zipfile, glob
from google.colab import files

uploaded = files.upload()                      # choose "Student folder.zip"
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/raw")

os.makedirs("data", exist_ok=True)
for split in ["train", "student_test"]:
    src = glob.glob(f"/content/raw/**/{split}", recursive=True)[0]
    if not os.path.exists(f"data/{split}"):
        shutil.move(src, f"data/{split}")
print({s: len(os.listdir(f"data/{s}")) for s in ["train", "student_test"]})

## 2. Exploratory data analysis
`data_processing/explore_data.py` creates a per-trip inventory: GPS points, time gaps, missing values, network size, connectivity and route information.

In [ ]:
!python -m data_processing.explore_data
import pandas as pd
pd.set_option("display.max_columns", 30)
trips = pd.read_csv("results/eda/trip_summary.csv", dtype={"trip_id": str})
trips.head(10)

In [ ]:
from data_processing.load_data import load_trip
t = load_trip("train", "00000005")
for k in ["track", "nodes", "arcs"]:
    print(k, t[k].shape, dict(t[k].dtypes))
    display(t[k].head(3))
print("route:", t["route"][:10], "... length", len(t["route"]))

## 3. Candidate generation, features and labels
For every GPS point we find nearby road arcs (within 50 m, up to 10) and compute features: distance, heading difference, speed and more.

The `.route` file lists arcs in driving order, not one per GPS point. So each point's label is the nearest route arc within 30 m. Points with no route arc nearby are left unlabelled.

In [ ]:
!python -m feature_engineering.build_dataset
cov = pd.read_csv("results/features/train_candidate_stats.csv", dtype={"trip_id": str})
cov.sort_values("label_coverage").head(8)

## 4. Model training: 5 models, 5-fold GroupKFold by trip
The models are Logistic Regression, Decision Tree, Gaussian Naive Bayes, KNN and Random Forest. KNN is slow, so the full run can take a while.

In [ ]:
!python -m model_training.train_models
cv = pd.read_csv("results/training/cv_summary.csv")
cv[["model", "macro_f1_mean", "macro_f1_std", "top1_accuracy_mean", "log_loss_mean", "train_macro_f1_mean", "train_time_s_mean"]].round(4)

In [ ]:
pd.read_csv("results/training/cv_folds.csv").round(4)

## 5. Student-test evaluation

In [ ]:
!python -m model_evaluation.evaluate_models
pd.read_csv("results/evaluation/student_test_metrics.csv").round(4)

## 6. HMM decoder tuning and Bayesian hyperparameter optimisation
The decoder penalties are tuned on 15 held-out **training** trips. Optuna TPE then searches the selected model's hyperparameters, with mean Macro F1 over GroupKFold as the objective.

In [ ]:
BEST_MODEL = "random_forest"   # chosen from the comparison above
!python -m hmm_viterbi.tune_decoder {BEST_MODEL}
pd.read_csv("results/hmm/decoder_grid.csv").head()

In [ ]:
!python -m hyperparameter_optimisation.bayes_opt {BEST_MODEL} 20
pd.read_csv(f"results/hpo/{BEST_MODEL}_untuned_vs_tuned.csv").drop(columns="params").round(4).T

## 7. HMM + Viterbi: classifier only vs classifier + HMM

In [ ]:
!python -m hmm_viterbi.evaluate_hmm
pd.read_csv("results/hmm/classifier_vs_hmm.csv").round(4)

## 8. Plots and Folium maps

In [ ]:
!python -m visualisations.plots
from IPython.display import Image, display
for f in sorted(glob.glob("results/plots/*.png")):
    print(f); display(Image(f))

In [ ]:
from visualisations.folium_maps import trip_map
trip_map("student_test", "00000052", f"{BEST_MODEL}_tuned")

## 9. API pipeline call
This is the same function the FastAPI `/predict` endpoint uses. Prediction never reads the route. Scoring reads it afterwards, on the server only.

In [ ]:
from api.pipeline import load_models, predict_trip, score_prediction
from hmm_viterbi.evaluate_hmm import load_params
models = load_models()
trip, points, arcs, clf_arcs, secs = predict_trip("student_test", "00000052", models[f"{BEST_MODEL}_tuned"], load_params())
print(f"inference {secs:.2f}s")
score_prediction("student_test", "00000052", trip, points, arcs, clf_arcs)